### Codigo 03 - Undersampling, Oversampling e SMOTE

Neste notebook, vamos aplicar tecnicas de reamostragem para lidar com dados desbalanceados.

A regra mais importante e: aplique undersampling, oversampling ou SMOTE somente no conjunto de treino. O conjunto de teste precisa continuar original para representar melhor o mundo real.

### 1. Instalacao do pacote necessario

As tecnicas RandomUnderSampler, RandomOverSampler e SMOTE ficam no pacote imbalanced-learn.

Se aparecer erro dizendo que imblearn nao existe, rode a linha abaixo em uma celula do notebook:

In [1]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


### 2. Importacoes

Agora importamos as bibliotecas. Vamos usar pandas, scikit-learn e imbalanced-learn.

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.model_selection import train_test_split

### 3. Configuracoes da base

Vamos trabalhar com a base Covertype. O arquivo esperado e cov_types.csv. Se ele nao existir, o notebook tentara baixar a base oficial da UCI.

In [3]:
CSV_PATH = Path("cov_types.csv")
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/covtype.data.gz"
RANDOM_STATE = 42
SAMPLE_SIZE = 30_000

COLUNAS = (
    [
        "Elevation",
        "Aspect",
        "Slope",
        "Horizontal_Distance_To_Hydrology",
        "Vertical_Distance_To_Hydrology",
        "Horizontal_Distance_To_Roadways",
        "Hillshade_9am",
        "Hillshade_Noon",
        "Hillshade_3pm",
        "Horizontal_Distance_To_Fire_Points",
    ]
    + [f"Wilderness_Area{i}" for i in range(1, 5)]
    + [f"Soil_Type{i}" for i in range(1, 41)]
    + ["Cover_Type"]
)

### 4. Funcao para carregar a base

Esta funcao evita repeticao de codigo. Ela carrega cov_types.csv se o arquivo existir. Caso contrario, baixa a base original da UCI e salva localmente.

In [4]:
def carregar_base() -> pd.DataFrame:
    """Carrega cov_types.csv ou baixa a base oficial."""
    if CSV_PATH.exists():
        print(f"Lendo arquivo local: {CSV_PATH.resolve()}")
        return pd.read_csv(CSV_PATH)
    
    print("Arquivo cov_types.csv nao encontrado.")
    print("Baixando a base oficial Covertype da UCI...")

    df = pd.read_csv(UCI_URL, header=None, names=COLUNAS, compression='gzip')
    df.to_csv(CSV_PATH, index=False)

    print(f"Arquivo salvo em: {CSV_PATH.resolve()}")
    return df

### 5. Carregando os dados e separando X e y

Agora carregamos a base e separamos as variaveis explicativas X da variavel alvo y.
A coluna alvo e Cover_Type, que representa a classe que queremos prever.

In [5]:
df = carregar_base()

print("Formato original:", df.shape)
display(df.head())

if len(df) > SAMPLE_SIZE:
    df =- df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE)
    print(f"Usando amostra de {SAMPLE_SIZE} linhas para treinar mais rápido.")

X = df.drop(columns="Cover_Type")
y = df['Cover_Type']

print("Formato de X", X.shape)
print("Formato de y", y.shape)


Lendo arquivo local: C:\Users\Olive\VS Code\Python\meu-projeto\machine_learning_and_Data_Science\9_dados_desbalanceados\cov_types.csv
Formato original: (581012, 55)


,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Wilderness_Area_1,Wilderness_Area_2,Wilderness_Area_3,Wilderness_Area_4,Soil_Type_1,Soil_Type_2,Soil_Type_3,Soil_Type_4,Soil_Type_5,Soil_Type_6,Soil_Type_7,Soil_Type_8,Soil_Type_9,Soil_Type_10,Soil_Type_11,Soil_Type_12,Soil_Type_13,Soil_Type_14,Soil_Type_15,Soil_Type_16,Soil_Type_17,Soil_Type_18,Soil_Type_19,Soil_Type_20,Soil_Type_21,Soil_Type_22,Soil_Type_23,Soil_Type_24,Soil_Type_25,Soil_Type_26,Soil_Type_27,Soil_Type_28,Soil_Type_29,Soil_Type_30,Soil_Type_31,Soil_Type_32,Soil_Type_33,Soil_Type_34,Soil_Type_35,Soil_Type_36,Soil_Type_37,Soil_Type_38,Soil_Type_39,Soil_Type_40,Cover_Type
0,2596,51,3,258,0,510,221,232,148,6279,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,5
1,2590,56,2,212,-6,390,220,235,151,6225,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,5
2,2804,139,9,268,65,3180,234,238,135,6121,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2
3,2785,155,18,242,118,3090,238,238,122,6211,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,2
4,2595,45,2,153,-1,391,220,234,150,6172,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,5


Usando amostra de 30000 linhas para treinar mais rápido.
Formato de X (30000, 54)
Formato de y (30000,)
